# Generacion de embeddings e indice FAISS (GPU)

Recibe un archivo **JSONL** consolidado (un chunk por linea, con campo `texto`),
genera los embeddings con `intfloat/multilingual-e5-base` y construye el indice
FAISS (`index.faiss` + `metadata.jsonl`) respetando el orden de insercion.

- Requiere `torch` con CUDA (y `sentence-transformers`, `faiss-cpu`, `numpy`).
- El prefijo de rol `passage: ` / `query: ` es obligatorio para E5.
- Disenado para corpus grandes: matriz en `np.memmap` + checkpoint reanudable.

In [ ]:
# ---------------------------------------------------------------------
# CUDA: deteccion de GPU y guia de instalacion de torch con CUDA
# ---------------------------------------------------------------------
import torch

print("Torch version  :", torch.__version__)
print("CUDA (torch)   :", torch.version.cuda)
print("CUDA available :", torch.cuda.is_available())

if torch.cuda.is_available():
    DEVICE = "cuda"
    print("GPU            :", torch.cuda.get_device_name(0))
    print("Capability     :", torch.cuda.get_capability(0))
    print("VRAM           :",
          round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1), "GB")
    torch.backends.cudnn.benchmark = True
else:
    DEVICE = "cpu"
    print("No hay GPU disponible. Si tu maquina tiene GPU NVIDIA, reinstala torch con CUDA:")
    print("  # CUDA 11.8:")
    print("  !pip uninstall torch torchvision torchaudio -y")
    print("  !pip install torch --index-url https://download.pytorch.org/whl/cu118")
    print("  # CUDA 12.1:")
    print("  !pip install torch --index-url https://download.pytorch.org/whl/cu121")
    print("  # (luego reinicia el kernel: Kernel -> Restart & Run All)")

In [ ]:
import os
import json
import math
from pathlib import Path

import numpy as np
import faiss
from sentence_transformers import SentenceTransformer
from tqdm import tqdm

# ---------------------------------------------------------------------
# Configuracion (DEVICE se define en la celda anterior)
# ---------------------------------------------------------------------
MODELO = "intfloat/multilingual-e5-base"
BATCH_SIZE = 64            # ajustar segun VRAM
USAR_FP16 = False          # encode en fp16 (solo CUDA); leve perdida de precision
PREFIJO_PASAJE = "passage: "
PREFIJO_CONSULTA = "query: "

RUTA_JSONL = "<ruta/al/corpus/consolidado/metadata.json>"
CARPETA_SALIDA = "<ruta/de/salida>"

NOMBRE_EMBEDDINGS = "embeddings.f32"
NOMBRE_CHECKPOINT = "checkpoint.json"

print(f"Modelo : {MODELO}")
print(f"Device : {DEVICE}")
print(f"Batch  : {BATCH_SIZE}")

In [ ]:
def leer_jsonl(ruta):
    """Lee un JSONL (un objeto por linea) y valida el campo 'texto'.

    Lectura en streaming: nunca carga el archivo completo con json.load,
    solo va acumulando los registros (cada chunk es pequeno).
    """
    ruta = Path(ruta)
    if not ruta.is_file():
        raise FileNotFoundError(f"No existe el archivo: {ruta}")

    registros = []
    with open(ruta, encoding="utf-8") as f:
        for n, linea in enumerate(f, start=1):
            linea = linea.strip()
            if not linea:
                continue
            obj = json.loads(linea)
            if not isinstance(obj, dict):
                raise TypeError(f"Linea {n}: se esperaba un objeto JSON.")
            if "texto" not in obj:
                raise KeyError(f"Linea {n}: falta el campo 'texto'. Claves: {sorted(obj)}")
            if not str(obj["texto"]).strip():
                raise ValueError(f"Linea {n}: texto vacio.")
            registros.append(obj)
    return registros

registros = leer_jsonl(RUTA_JSONL)
N = len(registros)
print(f"Chunks cargados: {N}")

campos = sorted({c for r in registros for c in r})
print(f"Campos detectados: {campos}")

In [ ]:
# ---------------------------------------------------------------------
# Modelo
# ---------------------------------------------------------------------
model = SentenceTransformer(MODELO, device=DEVICE)
DIM = model.get_sentence_embedding_dimension()
MAX_TOKENS = int(getattr(model, "max_seq_length", 512))
print(f"Dimension embedding : {DIM}")
print(f"Max seq length      : {MAX_TOKENS}")

# ---------------------------------------------------------------------
# Estimacion de memoria (RAM para el indice FAISS; la matriz va a disco)
# ---------------------------------------------------------------------
bytes_indice = N * DIM * 4
print(f"Tamano estimado del indice FAISS: {bytes_indice / 1024**2:.0f} MB "
      f"(~{bytes_indice / 1024**3:.2f} GB)")

os.makedirs(CARPETA_SALIDA, exist_ok=True)
ruta_emb = os.path.join(CARPETA_SALIDA, NOMBRE_EMBEDDINGS)
ruta_ckpt = os.path.join(CARPETA_SALIDA, NOMBRE_CHECKPOINT)

# ---------------------------------------------------------------------
# Checkpoint (reanudable)
# ---------------------------------------------------------------------
def cargar_checkpoint():
    if not os.path.exists(ruta_ckpt):
        return None
    with open(ruta_ckpt, encoding="utf-8") as f:
        ck = json.load(f)
    if ck.get("modelo") != MODELO or ck.get("N") != N or ck.get("dim") != DIM:
        print("Checkpoint incompatible (cambio modelo/N/dim); se reinicia.")
        return None
    return ck

ck = cargar_checkpoint()
inicio_bloque = ck["ultimo_bloque"] + 1 if ck else 0
total_bloques = math.ceil(N / BATCH_SIZE)
print(f"Bloques totales: {total_bloques} | retoma en: {inicio_bloque}")

# ---------------------------------------------------------------------
# Matriz destino en disco (memmap) y codificacion por bloques
# ---------------------------------------------------------------------
matriz = np.memmap(
    ruta_emb,
    mode="w+" if inicio_bloque == 0 else "r+",
    dtype="float32",
    shape=(N, DIM),
)

textos = [str(r["texto"]) for r in registros]

with tqdm(total=total_bloques, initial=inicio_bloque,
          desc="Embeddings", unit="bloque") as bar:
    for b in range(inicio_bloque, total_bloques):
        ini = b * BATCH_SIZE
        fin = min((b + 1) * BATCH_SIZE, N)
        lote = [PREFIJO_PASAJE + t for t in textos[ini:fin]]

        if USAR_FP16 and DEVICE == "cuda":
            with torch.autocast(device_type="cuda", dtype=torch.float16):
                vec = model.encode(
                    lote, batch_size=BATCH_SIZE, convert_to_numpy=True,
                    normalize_embeddings=True, show_progress_bar=False,
                )
        else:
            vec = model.encode(
                lote, batch_size=BATCH_SIZE, convert_to_numpy=True,
                normalize_embeddings=True, show_progress_bar=False,
            )

        matriz[ini:fin] = np.ascontiguousarray(vec, dtype="float32")
        matriz.flush()

        with open(ruta_ckpt, "w", encoding="utf-8") as f:
            json.dump({"modelo": MODELO, "N": N, "dim": DIM,
                       "ultimo_bloque": b}, f)
        bar.update(1)

print("Codificacion completa.")

In [ ]:
# ---------------------------------------------------------------------
# Indice FAISS (busqueda exacta; IP con vectores normalizados = coseno)
# ---------------------------------------------------------------------
index = faiss.IndexFlatIP(DIM)
BLOQUE_ADD = 10000  # anadir por bloques para no duplicar la matriz en RAM

with tqdm(total=math.ceil(N / BLOQUE_ADD), desc="FAISS", unit="bloque") as bar:
    for ini in range(0, N, BLOQUE_ADD):
        fin = min(ini + BLOQUE_ADD, N)
        index.add(np.ascontiguousarray(matriz[ini:fin], dtype="float32"))
        bar.update(1)

assert index.ntotal == N, f"Descuadre indice/metadata: {index.ntotal} != {N}"
print(f"Indice FAISS: {index.ntotal} vectores, dim={index.d}")

# ---------------------------------------------------------------------
# Persistencia (index.faiss + metadata.jsonl en el MISMO orden)
# ---------------------------------------------------------------------
ruta_index = os.path.join(CARPETA_SALIDA, "index.faiss")
ruta_metadata = os.path.join(CARPETA_SALIDA, "metadata.jsonl")

faiss.write_index(index, ruta_index)
with open(ruta_metadata, "w", encoding="utf-8") as f:
    for r in registros:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

print(f"Guardado: {ruta_index}")
print(f"Guardado: {ruta_metadata}")

# Opcional: liberar disco borrando la matriz intermedia (ya quedo en FAISS)
# os.remove(ruta_emb)

In [ ]:
# ---------------------------------------------------------------------
# Smoke test: recuperacion de ejemplo
# ---------------------------------------------------------------------
consulta = "inteligencia artificial y capacidades estrategicas"
vec = model.encode(
    [PREFIJO_CONSULTA + consulta], convert_to_numpy=True,
    normalize_embeddings=True, show_progress_bar=False,
)
vec = np.ascontiguousarray(vec, dtype="float32").reshape(1, -1)

k = 5
distancias, ids = index.search(vec, k)
print(f"Consulta: {consulta!r}")
for rank, (idx, dist) in enumerate(zip(ids[0], distancias[0]), start=1):
    r = registros[int(idx)]
    print(f"{rank}. score={dist:.4f} | {r['chunk_id']} | {str(r['texto'])[:120]}")